In [ ]:
from promenade.models import *

# Extractor tools

In [8]:
WEB_TOOLS = WebTools(READER_URL)

In [9]:
# res = await WEB_TOOLS.fetch_all_museum_data("https://victorymuseum.ru/")

In [10]:
EXTRACTPR_SYSTEM = """
You are an AI assistant that extracts and structures visitor information from website content. Your output must be a clean, well-organized report in Russian, even if the input is messy or lacks JavaScript-generated content.

Follow these rules strictly:

1. **Identify all distinct physical locations** (museums, branches, exhibition halls) mentioned in the text.  
   - If there are multiple locations (e.g., main museum + house-museum), create a separate section for each.  
   - Separate sections with a line of exactly 4 equals signs: `====`  
   - Order sections by importance (main location first, then branches).

2. **For each location, extract the following categories** (if available):  
   - `Часы работы` (opening hours) – list days and times clearly.  
   - `Стоимость билетов` (ticket prices) – any numbers, or mention if free, or "не указано".  
   - `Актуальные выставки и события` (current exhibitions & events) – include dates and short descriptions.  
   - `Специальные предложения` (special offers) – e.g., free admission days, discounts, package tickets.  
   - `Адрес и контакты` (address & contacts) – any physical address, phone, email, website.  
   - `Дополнительная информация` (additional info) – anything else useful: accessibility, educational programs, virtual tours, etc.

3. **If a category has no information** in the provided text, write `— информация отсутствует —` (do not invent data).

4. **Preserve concrete facts** (dates, times, prices, names).  
   - Do not paraphrase numbers or dates incorrectly.  
   - If a date is relative (“ближайшее бесплатное посещение — 16 апреля”), keep it as is.

5. **Be thorough** – scan the entire text for any minor detail that could affect a visitor’s decision (e.g., “музей сегодня работает до 21:00”, “требуется предварительный билет”, “Пушкинская карта”).

6. **Output structure** (example for one location):

   ## [Название места]
   **Часы работы**  
   - понедельник: 10:00–18:00  
   - ...

   **Стоимость билетов**  
   - ...

   **Актуальные выставки и события**  
   - [Название] (даты): описание...

   **Специальные предложения**  
   - ...

   **Адрес и контакты**  
   - ...

   **Дополнительная информация**  
   - ...

   ==== (if another location follows)

7. **Language:** The entire report must be in Russian, except for the separators (`====`). Use proper Russian punctuation and formatting.

8. **If the input is very short or contains only “У вас отключен JavaScript”** – still extract any hours, addresses, or links that are visible. Do not say “no information” if something is present.

Now produce the report based on the user’s input.
"""

In [11]:
# with open("res.txt", "w", encoding="utf8") as f:
#     textjsn = json.dumps(res)
#     f.write(textjsn)

# Reranking

In [12]:
with open("res.txt", "r", encoding="utf8") as f:
    text = "".join(f.readlines())
    res = json.loads(text)

In [ ]:
from promenade.models import RetreiveReranker, async_reranker_client, RERANKER_BASE_URL, QWEN_API_KEY

RERANKER_MODEL = "Qwen/Qwen3-Reranker-0.6B"
reranker = RetreiveReranker(
    rerank_n=1000, 
    retrieve_n=10000, 
    rerank_model=RERANKER_MODEL)

In [14]:
text_values = list(res.values())

In [15]:
text_values = reranker.rechank_docs(text_values)


In [16]:
resx =  await WEB_TOOLS.rerank_and_filter(text_values, query=EXTRACTPR_SYSTEM, reranker = reranker)

100%|██████████| 18/18 [00:05<00:00,  3.22it/s]


In [17]:
def mask_binary_urls(text: str) -> str:
    def replace_if_binary(match: re.Match) -> str:
        url = match.group(0)
        path = match.group(3).split('?')[0]
        if any(path.lower().endswith(ext) for ext in BINARY_FILES):
            return '[BIN URL]'
        return url

    return re.sub(URL_PATTERN, replace_if_binary, text)

In [18]:
len(mask_binary_urls(''.join(resx)))

540399

In [66]:
resy = mask_binary_urls(''.join(resx))

# Build extractor agend

In [30]:
def run_extractor_agent(user_message: str, tracer: ToolTracer) -> str:

    system_msg = SystemMessage(content=(EXTRACTPR_SYSTEM))
    messages = [system_msg, HumanMessage(content=user_message)]

    response = llm_chat(messages=messages)
    return response

In [ ]:
_t1a = ToolTracer()
resz = run_extractor_agent(''.join(resy), tracer=_t1a)

In [ ]:
print(resz.content)

## Музей Победы

**Часы работы**  
Сегодня работаем с 10:00 до 21:00

**Стоимость билетов**  
— информация отсутствует —

**Актуальные выставки и события**  
- **Акция «Георгиевская ленточка»** (23.04.2026 — 09.05.2026): В преддверии Дня Победы в Москве и по всей стране раздают символы памяти — Георгиевские ленточки.  
- **Субботник в Музее Г.О.Р.А.** (25.04.2026, 09:30): Совместный экологический субботник «Зелёная Весна».  
- **Концертная программа** (26.04.2026): В рамках конкурса «Журавли Победы» выступят Академический оркестр русских народных инструментов Радио и Телевидения, фольклорный ансамбль «Тараторки» и другие коллективы.  
- **«Защитникам непокоренного города-героя Севастополя посвящается!»** (28.04.2026, 17:00): Литературно-музыкальное патриотическое мероприятие в Зале Полководцев.  
- **«Красота спасает мир»** (05.03.2026 — 04.10.2026): Проект, посвящённый Международному женскому дню, где цветы представлены как универсальный язык эмоций и красоты.  
- **Старт голосования 

In [276]:
with open("test.txt", "w", encoding="utf8") as f:
    f.write(text)

In [34]:
print(''.join(resx))

Title: Музей Победы

URL Source: https://victorymuseum.ru/

Markdown Content:
Музей Победы

![Image 1](https://mc.yandex.ru/watch/17445043)![Image 2](https://mc.yandex.ru/watch/69747124)

Уважаемые посетители Музея Победы!  
  
Просим вас пройти опрос о качестве и удовлетворенности работой музея.  
  
Нам важно ваше мнение!  
  
[Пройти опрос](https://forms.mkrf.ru/e/2581/YrrHrJbi/?ap_orgcode=110200)

[![Image 3: Музей Победы](https://victorymuseum.ru/local/templates/white2/img/logo.svg)](https://victorymuseum.ru/)

*   [![Image 4](https://victorymuseum.ru/images/park-tickets-couple.svg)Билеты](https://victorymuseum.ru/for-visitors/prices/)
*   [Афиша](https://victorymuseum.ru/playbill/)
*   [Онлайн](https://victorymuseum.ru/online-programs/)
*   [Музей](https://victorymuseum.ru/about/)
*   [Проекты](https://victorymuseum.ru/projects/)
*   [Контакты](https://victorymuseum.ru/about/contacts/)
*   ![Image 5: Поиск по сайту](https://victorymuseum.ru/local/templates/white2/img/icon-search.